# Unified Dataset Transformation Workflow

This notebook merges and cleans the workflows from `datasets-merging.ipynb` and `transformer-500k.ipynb`.

Goal: document how raw multi-source text datasets are transformed into age-labeled parquet datasets for the project (age-group text classification).


## What This Notebook Covers

1. Common setup and reusable helpers.
2. Reddit (ABCDE) TSV -> cleaned parquet + balanced subsets.
3. Twitter/X (ABCDE city + country) TSV -> cleaned parquet + merged dataset.
4. Blog Authorship merged parquet -> inspection and examples.
5. PAN13 / PAN15 / PAN16 / PAN19 saved parquets -> quick EDA.
6. Hippocorpus CSV -> age-binned long-text parquet.
7. Final sanity checks and project-level summary.


In [ ]:
from pathlib import Path
import json
import math
import sys
import warnings
import xml.etree.ElementTree as ET

import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.parquet as pq
from IPython.display import display
from tqdm.auto import tqdm


# Make the repo-level `utils` package importable when the notebook is launched
# from `age/`, `complexity/`, or the project root.
PROJECT_ROOT = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / "utils").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the project root containing `utils/`.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.tasks.age import (
    AGE_BINS,
    AGE_LABELS,
    MODEL_CLASSES,
    PAN_NOTES,
    PAN_PARQUETS,
    SEED,
    LABEL_MAPPING_CONFIG,
    STANDARDIZED_BASE_DIR,
    UNIFIED_SCHEMA_COLUMNS,
    first_existing_path,
    ensure_parent_dir,
    age_distribution_parquet,
    process_tsv_to_parquet,
    sql_quote,
    sql_age_order,
    pan_quick_eda,
    map_to_5_classes,
    read_tabular_file,
    load_tabular_files,
    load_blog_raw,
    load_hippocorpus_raw,
    extract_pan14_text,
    load_pan14_raw,
    standardize_to_unified_schema,
    save_standardized_dataset,
)


In [ ]:
# Utilities moved to utils/tasks/age.py.


## 1) Reddit (ABCDE): TSV -> cleaned parquet

In [ ]:
reddit_raw = first_existing_path([
    "data/abcde/reddit/reddit_users_posts.tsv",
    "data/reddit/reddit_users_posts.tsv",
])

reddit_processed = Path("data/abcde/reddit/other/reddit_processed.parquet")

reddit_processed = process_tsv_to_parquet(
    input_path=reddit_raw,
    output_path=reddit_processed,
    usecols=["PostTitle", "PostSelftext", "DMGAgeAtPost"],
    text_builder=lambda c: c["PostTitle"].fillna("").astype(str) + ". " + c["PostSelftext"].fillna("").astype(str),
    age_col="DMGAgeAtPost",
    chunk_size=100_000,
)

print("Saved:", reddit_processed)
age_distribution_parquet(reddit_processed)


## 2) Reddit balanced subsets (from transformer workflow)

In [ ]:
con = duckdb.connect(database=":memory:")
con.execute(f"SELECT setseed({SEED})")

reddit_100k_per_class = reddit_processed.parent / "reddit_processed_100k_per_class.parquet"
reddit_200k_per_class = reddit_processed.parent / "reddit_processed_200k_per_class.parquet"

for per_class, out_path in [(100_000, reddit_100k_per_class), (200_000, reddit_200k_per_class)]:
    query = f"""
    COPY (
      SELECT age, text
      FROM (
        SELECT
          age,
          text,
          ROW_NUMBER() OVER (PARTITION BY age ORDER BY RANDOM()) AS rn
        FROM read_parquet('{reddit_processed}')
        WHERE age IN ({', '.join(f"'{c}'" for c in MODEL_CLASSES)})
          AND text IS NOT NULL
          AND length(text) > 0
      )
      WHERE rn <= {per_class}
    )
    TO '{out_path}'
    (FORMAT PARQUET, COMPRESSION ZSTD);
    """
    con.execute(f"SELECT setseed({SEED})")
    con.execute(query)
    print("Saved:", out_path)

con.close()


In [ ]:
print("100k/class distribution")
age_distribution_parquet(reddit_100k_per_class)


## 3) Reddit learning-curve subsets (1k -> 500k)

In [ ]:
sizes = [1_000, 5_000, 10_000, 50_000, 100_000, 500_000]
output_dir = Path("data/abcde/reddit")
output_dir.mkdir(parents=True, exist_ok=True)

max_per_class = max(sizes) // len(MODEL_CLASSES)

con = duckdb.connect(database=":memory:")
con.execute(f"SELECT setseed({SEED})")

con.execute(f"""
    CREATE TABLE reddit_pool AS
    SELECT text, age
    FROM (
        SELECT text, age,
               ROW_NUMBER() OVER (PARTITION BY age ORDER BY RANDOM()) AS rn
        FROM read_parquet('{reddit_processed}')
        WHERE age IN ({', '.join(f"'{c}'" for c in MODEL_CLASSES)})
    )
    WHERE rn <= {max_per_class}
""")

for size in sizes:
    per_class = size // len(MODEL_CLASSES)
    con.execute(f"SELECT setseed({SEED})")
    df = con.execute(f"""
        SELECT text, age
        FROM (
            SELECT text, age,
                   ROW_NUMBER() OVER (PARTITION BY age ORDER BY RANDOM()) AS rn
            FROM reddit_pool
        )
        WHERE rn <= {per_class}
        ORDER BY RANDOM()
    """).df()

    out_path = output_dir / f"reddit_{size // 1000}k.parquet"
    df.to_parquet(out_path, index=False)
    print(f"Saved {out_path} ({len(df):,} rows)")

con.close()


## 4) Twitter/X (ABCDE): city + country TSV -> cleaned parquet -> merged parquet

In [ ]:
twitter_city_raw = first_existing_path([
    "data/abcde/twitter/other/city_user_posts.tsv",
    "data/abcde/tusc/city_user_posts.tsv",
])

twitter_country_raw = first_existing_path([
    "data/abcde/twitter/other/country_user_posts.tsv",
    "data/abcde/tusc/country_user_posts.tsv",
])

twitter_city_parquet = Path("data/abcde/twitter/other/tusc_processed.parquet")
twitter_country_parquet = Path("data/abcde/twitter/other/country_user_posts.parquet")
twitter_merged_parquet = Path("data/abcde/twitter/other/tusc_merged.parquet")

process_tsv_to_parquet(
    input_path=twitter_city_raw,
    output_path=twitter_city_parquet,
    usecols=["PostText", "DMGAgeAtPost"],
    text_builder=lambda c: c["PostText"],
    age_col="DMGAgeAtPost",
    chunk_size=100_000,
)

process_tsv_to_parquet(
    input_path=twitter_country_raw,
    output_path=twitter_country_parquet,
    usecols=["PostText", "DMGAgeAtPost"],
    text_builder=lambda c: c["PostText"],
    age_col="DMGAgeAtPost",
    chunk_size=100_000,
)

twitter_city_df = pd.read_parquet(twitter_city_parquet)
twitter_country_df = pd.read_parquet(twitter_country_parquet)

twitter_merged_df = pd.concat([twitter_city_df, twitter_country_df], ignore_index=True)
twitter_merged_df.to_parquet(twitter_merged_parquet, index=False)

print("Saved:", twitter_merged_parquet)
age_distribution_parquet(twitter_merged_parquet)


In [ ]:
con = duckdb.connect(database=":memory:")
con.execute(f"SELECT setseed({SEED})")

count_13_17 = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{twitter_merged_parquet}') WHERE age = '13-17'
""").fetchone()[0]

per_other = max((500_000 - count_13_17) // 4, 0)
twitter_500k = Path("data/abcde/twitter/twitter_500k.parquet")

query = f"""
COPY (
    SELECT text, age
    FROM (
        SELECT text, age,
               ROW_NUMBER() OVER (PARTITION BY age ORDER BY RANDOM()) AS rn
        FROM read_parquet('{twitter_merged_parquet}')
        WHERE age IN ({', '.join(f"'{c}'" for c in MODEL_CLASSES)})
    )
    WHERE (age = '13-17')
       OR (age != '13-17' AND rn <= {per_other})
    ORDER BY RANDOM()
)
TO '{twitter_500k}'
(FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(query)
con.close()

print("Saved:", twitter_500k)
age_distribution_parquet(twitter_500k)


## 5) Blog Authorship: inspect merged blog parquet

In [10]:
blog_path = Path("data/blog/blog.parquet")
blog_df = pd.read_parquet(blog_path).copy()

blog_df["text"] = blog_df["text"].fillna("").astype(str)
blog_df["age"] = blog_df["age"].astype(str)
blog_df = blog_df[blog_df["age"].isin(MODEL_CLASSES)].reset_index(drop=True)

blog_df["char_len"] = blog_df["text"].str.len()
blog_df["word_len"] = blog_df["text"].str.split().map(len)
blog_df["preview"] = blog_df["text"].str.replace(r"\s+", " ", regex=True).str.slice(0, 180)

present_classes = [c for c in MODEL_CLASSES if c in set(blog_df["age"])]

print("Loaded:", blog_path)
print(f"Rows: {len(blog_df):,}")
print(f"Empty texts: {(blog_df['char_len'] == 0).sum():,}")
print("Classes:", ", ".join(present_classes))

blog_df[["age", "char_len", "word_len", "preview"]].head()

Loaded: data/blog/blog.parquet
Rows: 681,284
Empty texts: 0
Classes: 13-17, 18-29, 30-49


,age,char_len,word_len,preview
0,13-17,489,73,In het kader van kernfusie op aarde: MAAK JE ...
1,30-49,609,102,I had an interesting conversation with my Dad...
2,30-49,595,98,"If anything, Korea is a country of extremes. ..."
3,30-49,551,90,Take a read of this news article from urlLink...
4,30-49,555,91,"Ah, the Korean language...it looks so difficu..."


In [11]:
from IPython.display import display

class_dist = blog_df["age"].value_counts().rename_axis("age").reset_index(name="count")
class_dist["age"] = pd.Categorical(class_dist["age"], categories=present_classes, ordered=True)
class_dist["percent"] = (class_dist["count"] / class_dist["count"].sum() * 100).round(2)
class_dist = class_dist.sort_values("age").reset_index(drop=True)

length_summary = (
    blog_df.groupby("age")
    .agg(
        count=("text", "size"),
        avg_chars=("char_len", "mean"),
        median_chars=("char_len", "median"),
        p90_chars=("char_len", lambda s: s.quantile(0.90)),
        max_chars=("char_len", "max"),
        avg_words=("word_len", "mean"),
        median_words=("word_len", "median"),
    )
    .round(2)
    .reset_index()
)
length_summary["age"] = pd.Categorical(length_summary["age"], categories=present_classes, ordered=True)
length_summary = length_summary.sort_values("age").reset_index(drop=True)

example_frames = []
for age in present_classes:
    sample_n = min(3, int((blog_df["age"] == age).sum()))
    sample = (
        blog_df.loc[blog_df["age"] == age, ["age", "char_len", "word_len", "preview"]]
        .sample(n=sample_n, random_state=42)
        .sort_values("char_len", ascending=False)
    )
    example_frames.append(sample)

examples_per_class = pd.concat(example_frames, ignore_index=True)

print("Class distribution")
display(class_dist)

print("Length summary by class")
display(length_summary)

print("Sample examples by class")
display(examples_per_class)

Class distribution


,age,count,percent
0,13-17,235867,34.62
1,18-29,321447,47.18
2,30-49,123970,18.20


Length summary by class


,age,count,avg_chars,median_chars,p90_chars,max_chars,avg_words,median_words
0,13-17,235867,379.94,448.0,557.0,46136,67.95,80.0
1,18-29,321447,399.98,488.0,584.0,1826,70.25,87.0
2,30-49,123970,403.01,491.0,597.0,1721,69.53,86.0


Sample examples by class


,age,char_len,word_len,preview
0,13-17,547,101,So I got my official letter from Baylor LIFE ...
1,13-17,476,83,"Today sucked. I wanted to call Kelly, but nev..."
2,13-17,284,49,"i wonder, is there a difference between lovin..."
3,18-29,492,94,The Clock Part 1: The Naked Dance Of Twilight...
4,18-29,485,93,MY NOTEBOOK KANA VIRUS!!!!!!!! that's y there...
5,18-29,251,34,School girl got suspended for showing off her...
6,30-49,644,98,AlphaCAM Insights is a quick view into the wo...
7,30-49,602,107,Barry Bonds is on pace to not only break the ...
8,30-49,558,105,"Yep, went down to the Ice Palace (yeah, that'..."


### PAN datasets: quick EDA of saved document-level parquets

This subsection inspects the saved `pan13`, `pan15`, `pan16`, and `pan19` parquet files without rebuilding them. It reports dataset size, class imbalance, simple text-length statistics, and a couple of text previews per class.


In [6]:
for dataset_name, parquet_path in PAN_PARQUETS.items():
    print("=" * 100)
    print(f"{dataset_name.upper()} :: {parquet_path}")

    if not parquet_path.exists():
        print("Missing file.")
        continue

    row_count = pq.ParquetFile(parquet_path).metadata.num_rows
    if row_count == 0:
        empty_meta = pd.DataFrame([
            {
                "dataset": dataset_name,
                "rows": 0,
                "present_classes": 0,
                "largest_class_percent": None,
                "file_size_mb": round(parquet_path.stat().st_size / (1024 * 1024), 4),
                "note": PAN_NOTES.get(dataset_name, ""),
            }
        ])
        display(empty_meta)
        continue

    meta_df, class_df, examples_df = pan_quick_eda(dataset_name, parquet_path)
    display(meta_df)
    display(class_df)
    display(examples_df)


PAN13 :: data/pan13/pan13.parquet


,dataset,rows,present_classes,largest_class_percent,file_size_mb,note
0,pan13,413555,3,55.63,109.67,English only; one row per <conversation>; mapp...


,age,count,percent,avg_chars,max_chars,empty_texts
0,13-17,27651,6.69,489.31,986,0.0
1,18-29,155853,37.69,411.09,2883,0.0
2,30-49,230051,55.63,473.35,1520,0.0


,age,char_len,preview
0,13-17,195,"Hiya, I'm 11years old and board !! I go to Hig..."
1,13-17,88,"<img class=""smiley"" src=""http://www.pan.net/sm..."
2,18-29,570,The Spyware Doctor 2010 is the excellent spywa...
3,18-29,579,</br>; <br />;<p>;Wireless Television for comp...
4,30-49,510,The fantasy world of Spanish Fairy tales and f...
5,30-49,572,"Nevertheless, there are benefits about studyin..."


PAN15 :: data/pan15/pan15.parquet


,dataset,rows,present_classes,largest_class_percent,file_size_mb,note
0,pan15,7453,2,74.98,0.4,English only; one row per <document>; kept onl...


,age,count,percent,avg_chars,max_chars,empty_texts
0,18-29,5588,74.98,61.15,163,0.0
1,30-49,1865,25.02,87.16,146,0.0


,age,char_len,preview
0,18-29,124,I sit with a boy in english he became my frien...
1,18-29,42,boooored out of my mind #wannasleepbutdont
2,30-49,51,@username that tweet was mainly for you ;-) ch...
3,30-49,70,"I am watching Person of Interest, Pilot (S01E0..."


PAN16 :: data/pan16/pan16.parquet
Missing file.


## 6) Hippocorpus: CSV -> age-binned long-text parquet

In [ ]:
hippo_csv = first_existing_path(["hippoCorpusV2.csv", "data/hippocorpus/hippoCorpusV2.csv"])
hippo_df = pd.read_csv(hippo_csv, usecols=["story", "annotatorAge"])
hippo_df = hippo_df.rename(columns={"story": "text", "annotatorAge": "age"})

hippo_df["age"] = pd.cut(pd.to_numeric(hippo_df["age"], errors="coerce"), bins=AGE_BINS, labels=AGE_LABELS, right=False)
hippo_df = hippo_df[["text", "age"]].dropna(subset=["age"]).copy()
hippo_df["text"] = hippo_df["text"].fillna("").astype(str)
hippo_df["age"] = hippo_df["age"].astype(str)

hippo_df["token_len"] = hippo_df["text"].str.split().map(len)
hippo_long_df = hippo_df.loc[hippo_df["token_len"] > 128, ["text", "age"]]

hippo_long_path = Path("data/hippocorpus/hippocorpus_long.parquet")
ensure_parent_dir(hippo_long_path)
hippo_long_df.to_parquet(hippo_long_path, index=False)

print("Saved:", hippo_long_path)
age_distribution_parquet(hippo_long_path)


## 7) Final summary table for transformed outputs

In [ ]:
final_outputs = {
    "reddit_processed": Path("data/abcde/reddit/other/reddit_processed.parquet"),
    "reddit_100k_per_class": Path("data/abcde/reddit/other/reddit_processed_100k_per_class.parquet"),
    "reddit_200k_per_class": Path("data/abcde/reddit/other/reddit_processed_200k_per_class.parquet"),
    "twitter_merged": Path("data/abcde/twitter/other/tusc_merged.parquet"),
    "twitter_500k": Path("data/abcde/twitter/twitter_500k.parquet"),
    "blog_long": Path("data/blog/blog_long.parquet"),
    "blog_short_medium": Path("data/blog/blog_short_medium.parquet"),
    "hippocorpus_long": Path("data/hippocorpus/hippocorpus_long.parquet"),
}

rows = []
for name, path in final_outputs.items():
    if path.exists():
        try:
            row_count = duckdb.query(f"SELECT COUNT(*) FROM read_parquet('{path}')").fetchone()[0]
            rows.append({"dataset": name, "path": str(path), "rows": row_count})
        except Exception as e:
            rows.append({"dataset": name, "path": str(path), "rows": f"error: {e}"})
    else:
        rows.append({"dataset": name, "path": str(path), "rows": "missing"})

pd.DataFrame(rows).sort_values("dataset").reset_index(drop=True)


## Notes on Corrections Applied

- Removed notebook-only `pip install ...` cell from the workflow.
- Standardized path handling and added fallbacks for legacy folder layouts.
- Added helper functions to avoid repeated copy-paste transformations.
- Added explicit parent-directory creation before writing parquet outputs.
- Kept deterministic sampling with a fixed random seed.
- Added final sanity checks so the notebook clearly demonstrates project-ready transformed datasets.


## 8) Dataset standardization to 5-class target

Draft, merge-ready standardization for `blog`, `hippocorpus`, and `pan14` into a single schema for downstream 5-class age prediction.


### Shared utilities: loaders, schema, mapping stub, and output persistence

In [ ]:
# Utilities moved to utils/tasks/age.py.


### Blog dataset (draft loader + standardization)

In [ ]:
blog_dataset_dir = Path("/Users/georgijkutivadze/projects/mago-text-scoring/age/data/blog")

blog_raw_df = load_blog_raw(blog_dataset_dir)
blog_std_df = standardize_to_unified_schema(
    blog_raw_df,
    dataset_name="blog",
    text_candidates=["text", "content", "post", "body", "story"],
    label_candidates=["age", "age_group", "label", "class", "target"],
    doc_id_candidates=["doc_id", "id", "post_id", "uid", "user_id", "author_id"],
)

blog_paths, blog_stats = save_standardized_dataset(
    "blog",
    blog_std_df,
    rows_loaded=len(blog_raw_df),
)

print("Saved:", blog_paths)
print(json.dumps(blog_stats, indent=2, ensure_ascii=False))
blog_std_df.head()


### Hippocorpus dataset (draft loader + standardization)

In [ ]:
hippocorpus_dataset_dir = Path("/Users/georgijkutivadze/projects/mago-text-scoring/age/data/hippocorpus")

hippocorpus_raw_df = load_hippocorpus_raw(hippocorpus_dataset_dir)
hippocorpus_std_df = standardize_to_unified_schema(
    hippocorpus_raw_df,
    dataset_name="hippocorpus",
    text_candidates=["story", "text", "content", "body", "post"],
    label_candidates=["annotatorAge", "age", "age_group", "label", "class", "target"],
    doc_id_candidates=["doc_id", "id", "storyid", "uid", "user_id", "author_id"],
)

hippocorpus_paths, hippocorpus_stats = save_standardized_dataset(
    "hippocorpus",
    hippocorpus_std_df,
    rows_loaded=len(hippocorpus_raw_df),
)

print("Saved:", hippocorpus_paths)
print(json.dumps(hippocorpus_stats, indent=2, ensure_ascii=False))
hippocorpus_std_df.head()


### PAN14 dataset (draft loader + standardization)

In [ ]:
pan14_dataset_dir = Path("/Users/georgijkutivadze/projects/mago-text-scoring/age/data/pan14")

pan14_raw_df = load_pan14_raw(pan14_dataset_dir)
pan14_std_df = standardize_to_unified_schema(
    pan14_raw_df,
    dataset_name="pan14",
    text_candidates=["text", "document", "content", "body"],
    label_candidates=["raw_label", "age_group", "age", "label", "class", "target"],
    doc_id_candidates=["doc_id", "author_id", "id", "uid"],
)

pan14_paths, pan14_stats = save_standardized_dataset(
    "pan14",
    pan14_std_df,
    rows_loaded=len(pan14_raw_df),
)

print("Saved:", pan14_paths)
print(json.dumps(pan14_stats, indent=2, ensure_ascii=False))
pan14_std_df.head()


### Sanity checks and merge

In [ ]:
standardized_frames = []
for dataset_name in ["blog", "hippocorpus", "pan14"]:
    parquet_path = STANDARDIZED_BASE_DIR / dataset_name / "data.parquet"
    csv_path = STANDARDIZED_BASE_DIR / dataset_name / "data.csv"

    if parquet_path.exists():
        df = pd.read_parquet(parquet_path)
    elif csv_path.exists():
        df = pd.read_csv(csv_path)
    else:
        warnings.warn(f"Missing standardized output for {dataset_name}")
        continue

    missing_cols = [c for c in UNIFIED_SCHEMA_COLUMNS if c not in df.columns]
    if missing_cols:
        raise ValueError(f"{dataset_name}: missing required columns {missing_cols}")

    standardized_frames.append(df[UNIFIED_SCHEMA_COLUMNS].copy())

if standardized_frames:
    all_merged_df = pd.concat(standardized_frames, ignore_index=True)
else:
    all_merged_df = pd.DataFrame(columns=UNIFIED_SCHEMA_COLUMNS)

print("Rows by source:")
if all_merged_df.empty:
    print("Merged dataframe is empty (expected until mapping rules are configured).")
else:
    print(all_merged_df.groupby("source").size().sort_values(ascending=False))

    print("\nClass distribution per source (label):")
    print(pd.crosstab(all_merged_df["source"], all_merged_df["label"], margins=True))

    print("\nClass distribution overall (label_name):")
    print(all_merged_df["label_name"].value_counts(dropna=False))

all_merged_path = STANDARDIZED_BASE_DIR / "all_merged.parquet"
all_merged_df.to_parquet(all_merged_path, index=False)
print("\nSaved merged output:", all_merged_path)

all_merged_df.head()
